In [2]:
import os
import numpy as np
import geopandas as gpd
import pandas as pd
from shapely.geometry import Point
#import rasterio
#from rasterio.mask import mask

In [3]:
project_root = os.path.dirname(os.path.dirname("test.ipynb"))
data_dir = os.path.join(project_root, "data")
raster_dir = os.path.join(data_dir, "rasters")
transient_dir = os.path.join(project_root, "transients")
output_dir = os.path.join(project_root, "outputs")

energy_raster_path = os.path.join(raster_dir, "GHS_BUILT_S_timeseries_points.gpkg")


shapefile_path = os.path.join(data_dir, "ne_10m_admin_0_countries.shp")  # may need other files rather than just shp?
raster_path = os.path.join(data_dir, "gpw_v4_population_density_rev11_2020_30_min.tif")
country_energy_path = os.path.join(data_dir, "Country Energy Data.xlsx")

In [4]:
energy_timeseries = gpd.read_file(energy_raster_path)

led_data = pd.read_excel(country_energy_path)
chosen_column = [str(col) for col in led_data.columns if "Chosen" in str(col)][0]
led_data = led_data[led_data[chosen_column] > 0].dropna(subset=[chosen_column])


year = 2025
place_ocean = True
all_leds_gdf = gpd.GeoDataFrame()

In [ ]:
country_name="South Korea"
surround = 1
values_filtered = energy_timeseries[energy_timeseries["country"] == country_name]



In [37]:
for index, row in led_data.iterrows():

    country_name = row['Entity']
    num_leds = int(row['Round'])
    leds_placed = 0

    values_array = energy_timeseries[energy_timeseries['country'] == country_name]
    values_array = values_array[["point_index", f"{year}", "geometry"]].sort_values(f"{year}", ascending=False)

    available_cells = len(values_array)
    missing_leds = num_leds - available_cells
    
    if available_cells == 0:
        print(f"Could not find {country_name} in raster data, skipping...")

    else:

        for leds in range(0,num_leds-leds_placed): # Place LEDs on the land-space

            if leds_placed < available_cells: 
                
                all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_array["geometry"].iloc[leds]],
                                                                        'Country': [country_name],
                                                                        'Raster_Density': [values_array[f"{year}"].iloc[leds]]
                                                                        }, geometry='geometry')], ignore_index=True)
                leds_placed += 1

            else:

                if place_ocean == True:
                    
                    if leds_placed >= num_leds:
                        break

                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {available_cells} cells are available. Attempting to place {missing_leds} LEDs in surrounding area.")
                    values_sorted = energy_timeseries.sort_values('point_index').reset_index(drop=True)
                    surround = 1
                    
                    while leds_placed < num_leds:

                        surround_indices = ([x - (720*surround) for x in values_array.point_index] + 
                                            [x + (720*surround) for x in values_array.point_index] +
                                            [x - surround for x in values_array.point_index] + 
                                            [x + surround for x in values_array.point_index]) 
                        surround_indices = np.unique(list(filter(lambda x: x >= 0, surround_indices)))
                        values_filtered = energy_timeseries[energy_timeseries["point_index"].isin(surround_indices)].query("country.isnull()")

                        for leds in range(len(values_filtered)): # Place remaining LEDs on the land-space
                            all_leds_gdf = pd.concat([all_leds_gdf, gpd.GeoDataFrame({'geometry': [values_filtered["geometry"].iloc[leds]],
                                                                                    'Country': [country_name],
                                                                                    'Raster_Density': [values_filtered[f"{year}"].iloc[leds]]
                                                                                    }, geometry='geometry')], ignore_index=True)
                            leds_placed += 1
                            if leds_placed >= num_leds:
                                break
                        surround += 1
                
                else: 
                    print(f"Warning: {country_name} requested {num_leds} LEDs, but only {leds_placed} were placed due to not having enough space.")
                    break

Could not find Hong Kong S.A.R. in raster data, skipping...
Could not find Bahrain in raster data, skipping...
Could not find Republic of Serbia in raster data, skipping...
Could not find Dominican Republic in raster data, skipping...
Could not find Bosnia and Herzegovina in raster data, skipping...
Could not find Ivory Coast in raster data, skipping...
Could not find United Republic of Tanzania in raster data, skipping...
Could not find Netherlands Antilles in raster data, skipping...


In [38]:
all_leds_gdf.to_file("dataframe.gpkg", driver="GPKG")

c:\ProgramData\miniforge3\envs\science_gen\Lib\site-packages\pyogrio\geopandas.py:710: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(
